In [61]:
import os
%pip install py7zr


In [62]:
%pwd

'C:\\Project\\CHICKEN-DISEASE-CLASSIFICATION-PROJECT'

In [63]:
os.chdir("../")

In [64]:
%pwd

'C:\\Project'

for entity file to provide data class to the config.yaml

In [65]:

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_path:Path


In [66]:
from cnnClassifier.constants import *

In [67]:

from cnnClassifier.utils.common import read_yaml, create_directories

after update entity we update configuration file in src config

In [68]:
from pathlib import  Path
from cnnClassifier.constants import *
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])        

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file= config.local_data_file,
            unzip_path =config.unzip_path
        )

        return data_ingestion_config


now we update components

In [69]:
import os
import zipfile
import urllib.request as request
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [ ]:
class Dataingestion:
    def __init__(self,config:DataIngestionConfig):
        self.config = config
        
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, header=request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} : download with following info: \n{header}")
        else:
            logger.info(f"file already exist of size:{get_size(Path(self.config.local_data_file))}")
            
            
    def extract_zip_file(self):
        unzip_path = self.config.unzip_path
        os.makedirs(unzip_path, exist_ok=True)
        with py7zr.SevenZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        
            

##create pipelining

In [76]:
import py7zr
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=Dataingestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


File: config\config.yaml
Content: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/itxhamzamaqsood/ml-project-image/raw/refs/heads/main/Chicken-fecal-images.7z', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_path': 'artifacts/data_ingestion'}}
Type: <class 'dict'>

File: params.yaml
Content: {'key': 'value'}
Type: <class 'dict'>
[2026-10-17 22:39:29,871: INFO: common: created directory at: artifacts]
[2026-10-17 22:39:29,875: INFO: common: created directory at: artifacts/data_ingestion]
[2026-10-17 22:39:29,879: INFO: 3264400367: file already exist of size:~ 10850 KB]
